# Scope 2 Carbon Emissions — Combined Forecasting Pipeline
Produces **three sets** of Power BI-ready CSVs in one run:
- **State-based** → `state_based/` folder
- **City-based**  → `city_based/` folder
- **Site-based**  → `site_based/` folder

Each granularity × each of the 3 target columns exports 4 CSVs:

| File pattern | Description |
|---|---|
| `2_model_performance_<target>_<level>.csv` | Backtest metrics per model & target |
| `2_forecast_long_<target>_<level>.csv` | Backtest historical + predicted (long format) |
| `2_forecast_best_model_<target>_<level>.csv` | Historical + best-model future forecast with CIs |
| `2_forecast_all_models_<target>_<level>.csv` | Historical + all-model future forecasts with CIs |


## Imports

In [37]:
import pandas as pd
import numpy as np
import os

from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet


## Configuration
All user-facing settings in one place.

In [38]:
# ── USER SETTINGS ────────────────────────────────────────────────────────────
INPUT_FILE       = "Scope 2 data Updated.csv"   # path to your CSV file
CUTOFF_DATE      = pd.Timestamp("2026-03-31")   # trim data after this date
FORECAST_HORIZON = 36                           # months into the future
BACKTEST_HORIZON = 6                            # months held-out for backtest
MIN_HISTORY      = 24                           # min months needed per group
N_KNN_NEIGHBORS  = 3                            # KNN imputation neighbours

TARGETS = [
    "total_carbon_emissions_kg",
    "dg_units_in_kwh",
    "total_units_kwh",
]

MODEL_LIST = ["Naive", "ARIMA", "HoltWinters", "Prophet", "SES", "Holt", "SARIMAX"]


## Shared Helpers

In [39]:
# ── 1. Basic cleaning ─────────────────────────────────────────────────────────
def basic_scope2_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ .+]", "_", regex=True)
        .str.replace(r"^-+",   "", regex=True)
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+",    "_", regex=True)
        .str.strip("_")
    )

    keep_cols = ["indicator_name", "state", "city", "site", "d_end_date", *TARGETS]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    if {"indicator_name", "state", "city", "site", "d_end_date"}.issubset(df.columns):
        for c in ["indicator_name", "state", "city", "site"]:
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c].isin(["", "nan", "none", "null"]), c] = np.nan

        df["d_end_date"] = pd.to_datetime(df["d_end_date"], errors="coerce")

        for c in TARGETS:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        num_cols     = df.select_dtypes(include=["number"]).columns.tolist()
        non_num_cols = [c for c in df.columns if c not in num_cols]
        df = (
            df.groupby(
                ["indicator_name", "state", "city", "site", "d_end_date"],
                as_index=False
            )
            .agg({
                **{c: "sum"   for c in num_cols},
                **{c: "first" for c in non_num_cols
                   if c not in ["indicator_name", "state", "city", "site", "d_end_date"]}
            })
        )

    if "d_end_date" in df.columns:
        df["d_end_date"] = pd.to_datetime(df["d_end_date"], errors="coerce")
        try:
            df["d_end_date"] = df["d_end_date"].dt.tz_localize(None)
        except Exception:
            pass
        df["date"] = df["d_end_date"].dt.to_period("M").dt.to_timestamp()

    return df


# ── 2. Evaluation metrics ──────────────────────────────────────────────────────
def evaluate(y_true, y_pred):
    mask   = ~(pd.isna(y_true) | pd.isna(y_pred))
    y_true = np.array(y_true)[mask]
    y_pred = np.array(y_pred)[mask]
    if len(y_true) == 0:
        return np.nan, np.nan, np.nan, np.nan
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mu   = np.mean(np.abs(y_true))
    if mu == 0:
        return mae, rmse, np.nan, np.nan
    return mae, rmse, (mae / mu) * 100, (rmse / mu) * 100


# ── 3. Fit & forecast a single model ─────────────────────────────────────────
def _fit_forecast(model_name: str, y_train, horizon: int):
    """Return (pred, lower_ci, upper_ci) arrays of length `horizon`."""
    pred     = np.full(horizon, np.nan)
    lower_ci = np.full(horizon, np.nan)
    upper_ci = np.full(horizon, np.nan)
    try:
        if model_name == "Naive":
            pred = np.maximum(np.repeat(y_train.iloc[-1], horizon), 0)

        elif model_name == "ARIMA":
            fc       = ARIMA(y_train, order=(1, 1, 1)).fit().get_forecast(steps=horizon)
            pred     = np.maximum(np.array(fc.predicted_mean), 0)
            ci       = fc.conf_int(alpha=0.05)
            lower_ci = np.maximum(ci.iloc[:, 0].to_numpy(), 0)
            upper_ci = np.maximum(ci.iloc[:, 1].to_numpy(), 0)

        elif model_name == "HoltWinters":
            fitted = ExponentialSmoothing(
                y_train, trend="add", seasonal="add", seasonal_periods=12).fit()
            pred = np.maximum(np.array(fitted.forecast(horizon)), 0)
            if hasattr(fitted, "resid") and fitted.resid is not None:
                std = np.nanstd(fitted.resid)
                if not np.isnan(std):
                    lower_ci = np.maximum(pred - 1.96 * std, 0)
                    upper_ci = pred + 1.96 * std

        elif model_name == "SES":
            pred = np.maximum(
                np.array(SimpleExpSmoothing(y_train).fit().forecast(horizon)), 0)

        elif model_name == "Holt":
            pred = np.maximum(
                np.array(Holt(y_train).fit().forecast(horizon)), 0)

        elif model_name == "SARIMAX":
            fc = SARIMAX(
                y_train, order=(1, 1, 1),
                seasonal_order=(1, 1, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False).get_forecast(steps=horizon)
            pred     = np.maximum(np.array(fc.predicted_mean), 0)
            ci       = fc.conf_int(alpha=0.05)
            lower_ci = np.maximum(ci.iloc[:, 0].to_numpy(), 0)
            upper_ci = np.maximum(ci.iloc[:, 1].to_numpy(), 0)

    except Exception:
        pass
    return pred, lower_ci, upper_ci


def _fit_forecast_prophet(prophet_df: pd.DataFrame, horizon: int):
    pred     = np.full(horizon, np.nan)
    lower_ci = np.full(horizon, np.nan)
    upper_ci = np.full(horizon, np.nan)
    try:
        m = Prophet(yearly_seasonality=True,
                    weekly_seasonality=False,
                    daily_seasonality=False)
        m.fit(prophet_df)
        future   = m.make_future_dataframe(periods=horizon, freq="MS")
        fc       = m.predict(future)
        pred     = np.maximum(fc["yhat"].iloc[-horizon:].to_numpy(), 0)
        lower_ci = np.maximum(fc["yhat_lower"].iloc[-horizon:].to_numpy(), 0)
        upper_ci = np.maximum(fc["yhat_upper"].iloc[-horizon:].to_numpy(), 0)
    except Exception:
        pass
    return pred, lower_ci, upper_ci


## Generic Pipeline Functions
Each function accepts `group_keys` — set automatically based on granularity (`state`, `city`, or `site`).

In [40]:
# ── Aggregate monthly ─────────────────────────────────────────────────────────
def aggregate_monthly(df: pd.DataFrame, value_col: str, group_keys: list) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    cols = group_keys + ["date"]
    return df.groupby(cols, as_index=False)[value_col].sum()


# ── Reindex to full monthly calendar per group ────────────────────────────────
def reindex_monthly(df: pd.DataFrame, value_col: str, group_keys: list) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    out = []
    for key_vals, g in df.groupby(group_keys):
        g = g.sort_values("date")
        if g["date"].isna().all():
            continue
        idx = pd.date_range(g["date"].min(), g["date"].max(), freq="MS")
        g   = g.set_index("date").reindex(idx)
        g.index.name = "date"
        g   = g.reset_index()
        vals = key_vals if isinstance(key_vals, tuple) else (key_vals,)
        for k, v in zip(group_keys, vals):
            g[k] = v
        out.append(g)
    if not out:
        return pd.DataFrame(columns=group_keys + ["date", value_col])
    return pd.concat(out, ignore_index=True)


# ── KNN imputation per group ──────────────────────────────────────────────────
def knn_impute(df: pd.DataFrame, value_col: str, group_keys: list,
               n_neighbors: int = N_KNN_NEIGHBORS) -> pd.DataFrame:
    df = df.copy()
    out = []
    for _, g in df.groupby(group_keys):
        g = g.sort_values("date").reset_index(drop=True)
        g["_t"] = np.arange(len(g))
        X = g[["_t", value_col]].values
        g[value_col] = KNNImputer(n_neighbors=n_neighbors).fit_transform(X)[:, 1]
        g = g.drop(columns=["_t"])
        out.append(g)
    if not out:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(out, ignore_index=True)


# ── Drop all-zero / all-missing groups ────────────────────────────────────────
def drop_all_zero_or_missing(df: pd.DataFrame, value_col: str, group_keys: list) -> pd.DataFrame:
    df = df.copy()
    stats = (
        df.groupby(group_keys)[value_col]
        .agg(total_rows="size",
             zero_count=lambda x: (x == 0).sum(),
             missing_count=lambda x: x.isna().sum())
        .reset_index()
    )
    stats["bad_rows"] = stats["zero_count"] + stats["missing_count"]
    bad = stats.loc[stats["bad_rows"] == stats["total_rows"], group_keys]
    merged = df.merge(bad, on=group_keys, how="left", indicator=True)
    return merged[merged["_merge"] == "left_only"].drop(columns="_merge")


# ── Full preprocessing pipeline ───────────────────────────────────────────────
def preprocess_pipeline(df_raw: pd.DataFrame, value_col: str,
                        group_keys: list) -> pd.DataFrame:
    df = basic_scope2_clean(df_raw)
    df = aggregate_monthly(df, value_col, group_keys)
    df = reindex_monthly(df, value_col, group_keys)
    df = df[df["date"] <= CUTOFF_DATE].copy()
    df = knn_impute(df, value_col, group_keys)
    df = drop_all_zero_or_missing(df, value_col, group_keys)
    return df


In [41]:
# ── Backtest: all models ──────────────────────────────────────────────────────
def run_backtest(df: pd.DataFrame, value_col: str, group_keys: list,
                 forecast_horizon: int = BACKTEST_HORIZON,
                 min_history: int = MIN_HISTORY):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    results, forecast_rows = [], []

    for key_vals, group in df.groupby(group_keys):
        group = group.sort_values("date").copy()
        if len(group) < min_history:
            continue
        train = group.iloc[:-forecast_horizon].copy()
        test  = group.iloc[-forecast_horizon:].copy()
        if len(train) == 0 or len(test) == 0:
            continue

        y_train    = train[value_col].reset_index(drop=True)
        y_test     = test[value_col].reset_index(drop=True)
        train_dates = train["date"].reset_index(drop=True)
        test_dates  = test["date"].reset_index(drop=True)
        if y_train.nunique() < 2:
            continue

        vals     = key_vals if isinstance(key_vals, tuple) else (key_vals,)
        key_dict = dict(zip(group_keys, vals))

        preds = {}
        for m in MODEL_LIST:
            if m == "Prophet":
                prophet_df = train[["date", value_col]].rename(
                    columns={"date": "ds", value_col: "y"})
                p, _, _ = _fit_forecast_prophet(prophet_df, len(test))
            else:
                p, _, _ = _fit_forecast(m, y_train, len(test))
            preds[m] = p

        metrics = [(m, *evaluate(y_test, p)) for m, p in preds.items()]
        valid   = [x for x in metrics if pd.notna(x[2])]
        if not valid:
            continue
        best = min(valid, key=lambda x: x[2])[0]

        for m, mae, rmse, mae_pct, rmse_pct in metrics:
            results.append({**key_dict, "target": value_col, "model": m,
                            "MAE": mae, "RMSE": rmse,
                            "MAE_%": mae_pct, "RMSE_%": rmse_pct,
                            "is_best": int(m == best)})

        for m in preds:
            # Historical rows
            for i in range(len(train)):
                forecast_rows.append({**key_dict, "target": value_col,
                                       "model": m, "date": train_dates.iloc[i],
                                       "series": "Historical", "value": y_train.iloc[i]})
            # Forecast rows
            for i in range(len(test)):
                forecast_rows.append({**key_dict, "target": value_col,
                                       "model": m, "date": test_dates.iloc[i],
                                       "series": "Forecast", "value": preds[m][i]})

    return pd.DataFrame(results), pd.DataFrame(forecast_rows)


# ── Future forecast: best model only ─────────────────────────────────────────
def forecast_best_only(df: pd.DataFrame, results_df: pd.DataFrame,
                       value_col: str, group_keys: list,
                       horizon: int = FORECAST_HORIZON):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    best_models = (
        results_df.loc[results_df["is_best"] == 1,
                       group_keys + ["target", "model"]]
        .rename(columns={"model": "best_model"})
        .drop_duplicates()
    )
    output_rows = []

    for key_vals, group in df.groupby(group_keys):
        group = group.sort_values("date").copy()
        y     = group[value_col].reset_index(drop=True)
        if len(group) < MIN_HISTORY or y.nunique() < 2:
            continue

        vals     = key_vals if isinstance(key_vals, tuple) else (key_vals,)
        key_dict = dict(zip(group_keys, vals))

        bm = best_models.copy()
        for k, v in key_dict.items():
            bm = bm[bm[k] == v]
        bm = bm[bm["target"] == value_col]
        if bm.empty:
            continue
        best_model = bm["best_model"].iloc[0]

        future_dates = pd.date_range(
            group["date"].max() + pd.offsets.MonthBegin(1),
            periods=horizon, freq="MS")

        if best_model == "Prophet":
            prophet_df = group[["date", value_col]].rename(
                columns={"date": "ds", value_col: "y"})
            pred, lower_ci, upper_ci = _fit_forecast_prophet(prophet_df, horizon)
        else:
            pred, lower_ci, upper_ci = _fit_forecast(best_model, y, horizon)

        for i in range(len(group)):
            output_rows.append({**key_dict, "target": value_col,
                                 "date": group["date"].iloc[i],
                                 "series": "Historical", "value": y.iloc[i],
                                 "model": best_model, "best_model": best_model,
                                 "lower_ci": np.nan, "upper_ci": np.nan})
        for i in range(horizon):
            output_rows.append({**key_dict, "target": value_col,
                                 "date": future_dates[i],
                                 "series": "Best Forecast", "value": pred[i],
                                 "model": best_model, "best_model": best_model,
                                 "lower_ci": lower_ci[i], "upper_ci": upper_ci[i]})

    return pd.DataFrame(output_rows)


# ── Future forecast: all models + historical ──────────────────────────────────
def forecast_all(df: pd.DataFrame, value_col: str, group_keys: list,
                 horizon: int = FORECAST_HORIZON):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    output_rows = []

    for key_vals, group in df.groupby(group_keys):
        group = group.sort_values("date").copy()
        y     = group[value_col].reset_index(drop=True)
        if len(group) < MIN_HISTORY or y.nunique() < 2:
            continue

        vals     = key_vals if isinstance(key_vals, tuple) else (key_vals,)
        key_dict = dict(zip(group_keys, vals))

        future_dates = pd.date_range(
            group["date"].max() + pd.offsets.MonthBegin(1),
            periods=horizon, freq="MS")

        # Historical rows (written once, labelled with series_type="Historical")
        for i in range(len(group)):
            output_rows.append({**key_dict, "target": value_col,
                                 "date": group["date"].iloc[i],
                                 "series_type": "Historical", "model": "Historical",
                                 "value": y.iloc[i],
                                 "lower_ci": np.nan, "upper_ci": np.nan})

        for model_name in MODEL_LIST:
            if model_name == "Prophet":
                prophet_df = group[["date", value_col]].rename(
                    columns={"date": "ds", value_col: "y"})
                pred, lower_ci, upper_ci = _fit_forecast_prophet(prophet_df, horizon)
            else:
                pred, lower_ci, upper_ci = _fit_forecast(model_name, y, horizon)

            for i in range(horizon):
                output_rows.append({**key_dict, "target": value_col,
                                     "date": future_dates[i],
                                     "series_type": "Forecast", "model": model_name,
                                     "value": pred[i],
                                     "lower_ci": lower_ci[i], "upper_ci": upper_ci[i]})

    return pd.DataFrame(output_rows)


## Run All Three Granularities
Loops over state → city → site, and within each level loops over all 3 target columns.

In [42]:
GK_MAP = {
    "state": ["indicator_name", "state"],
    "city":  ["indicator_name", "state", "city"],
    "site":  ["indicator_name", "state", "city", "site"],
}

def run_level(df_raw: pd.DataFrame, granularity: str) -> None:
    """Run full pipeline for one granularity and write CSVs to its subfolder."""
    group_keys = GK_MAP[granularity]
    output_dir = f"{granularity}_based"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  {granularity.upper()}-based pipeline")
    print(f"{'='*60}")

    for value_col in TARGETS:
        safe_col = value_col.lower().replace("/", "_").replace(" ", "_")
        print(f"  \u25b8 target: {value_col}")

        df = preprocess_pipeline(df_raw, value_col, group_keys)
        print(f"    preprocessed rows : {len(df):,}")

        results_df, forecast_long_df = run_backtest(
            df, value_col, group_keys,
            forecast_horizon=BACKTEST_HORIZON, min_history=MIN_HISTORY)
        print(f"    backtest rows     : {len(results_df):,} metrics  |  {len(forecast_long_df):,} forecast")

        future_best_df = forecast_best_only(
            df, results_df, value_col, group_keys, horizon=FORECAST_HORIZON)

        future_all_df = forecast_all(
            df, value_col, group_keys, horizon=FORECAST_HORIZON)

        # Save per-target files
        for df_, fname in [
            (results_df,       f"2_model_performance_{safe_col}_{granularity}_based.csv"),
            (forecast_long_df, f"2_forecast_long_{safe_col}_{granularity}_based.csv"),
            (future_best_df,   f"2_forecast_best_model_{safe_col}_{granularity}_based.csv"),
            (future_all_df,    f"2_forecast_all_models_{safe_col}_{granularity}_based.csv"),
        ]:
            path = os.path.join(output_dir, fname)
            df_.to_csv(path, index=False)
            print(f"  \u2713  {path}")

    print(f"  Done \u2014 {output_dir}/ written.")


In [ ]:
# ── MAIN EXECUTION ────────────────────────────────────────────────────────────
df_raw = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_raw):,} rows from '{INPUT_FILE}'")

for level in ["state", "city", "site"]:
    run_level(df_raw, level)

print("\nAll done. CSVs are in state_based/, city_based/, site_based/")


19:01:06 - cmdstanpy - INFO - Chain [1] start processing
19:01:06 - cmdstanpy - INFO - Chain [1] done processing


Loaded 4,050 rows from 'Scope 2 data Updated.csv'

  STATE-based pipeline
  ▸ target: total_carbon_emissions_kg
    preprocessed rows : 1,411


19:01:06 - cmdstanpy - INFO - Chain [1] start processing
19:01:06 - cmdstanpy - INFO - Chain [1] done processing
19:01:06 - cmdstanpy - INFO - Chain [1] start processing
19:01:06 - cmdstanpy - INFO - Chain [1] done processing
19:01:07 - cmdstanpy - INFO - Chain [1] start processing
19:01:07 - cmdstanpy - INFO - Chain [1] done processing
19:01:07 - cmdstanpy - INFO - Chain [1] start processing
19:01:07 - cmdstanpy - INFO - Chain [1] done processing
19:01:07 - cmdstanpy - INFO - Chain [1] start processing
19:01:07 - cmdstanpy - INFO - Chain [1] done processing
19:01:07 - cmdstanpy - INFO - Chain [1] start processing
19:01:07 - cmdstanpy - INFO - Chain [1] done processing
19:01:07 - cmdstanpy - INFO - Chain [1] start processing
19:01:07 - cmdstanpy - INFO - Chain [1] done processing
19:01:08 - cmdstanpy - INFO - Chain [1] start processing
19:01:08 - cmdstanpy - INFO - Chain [1] done processing
19:01:08 - cmdstanpy - INFO - Chain [1] start processing
19:01:08 - cmdstanpy - INFO - Chain [1]

    backtest rows     : 105 metrics  |  9,877 forecast


19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:10 - cmdstanpy - INFO - Chain [1] start processing
19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:10 - cmdstanpy - INFO - Chain [1] start processing
19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:10 - cmdstanpy - INFO - Chain [1] start processing
19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:10 - cmdstanpy - INFO - Chain [1] start processing
19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:10 - cmdstanpy - INFO - Chain [1] start processing
19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:10 - cmdstanpy - INFO - Chain [1] start processing
19:01:10 - cmdstanpy - INFO - Chain [1] done processing
19:01:11 - cmdstanpy - INFO - Chain [1] start processing
19:01:11 - cmdstanpy - INFO - Chain [1] done processing
19:01:11 - cmdstanpy - INFO - Chain [1] start processing
19:01:11 - cmdstanpy - INFO - Chain [1] done processing
19:01:11 - cmdstanpy - INFO - Chain [1] 

...

19:02:19 - cmdstanpy - INFO - Chain [1] start processing
19:02:19 - cmdstanpy - INFO - Chain [1] done processing
19:02:20 - cmdstanpy - INFO - Chain [1] start processing
19:02:20 - cmdstanpy - INFO - Chain [1] done processing
19:02:20 - cmdstanpy - INFO - Chain [1] start processing
19:02:20 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
19:02:20 - cmdstanpy - INFO - Chain [1] start processing
19:02:20 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/s

    backtest rows     : 231 metrics  |  24,010 forecast


19:02:29 - cmdstanpy - INFO - Chain [1] start processing
19:02:29 - cmdstanpy - INFO - Chain [1] done processing
19:02:29 - cmdstanpy - INFO - Chain [1] start processing
19:02:29 - cmdstanpy - INFO - Chain [1] done processing
19:02:29 - cmdstanpy - INFO - Chain [1] start processing
19:02:29 - cmdstanpy - INFO - Chain [1] done processing
19:02:29 - cmdstanpy - INFO - Chain [1] start processing
19:02:29 - cmdstanpy - INFO - Chain [1] done processing
19:02:29 - cmdstanpy - INFO - Chain [1] start processing
19:02:29 - cmdstanpy - INFO - Chain [1] done processing
19:02:30 - cmdstanpy - INFO - Chain [1] start processing
19:02:30 - cmdstanpy - INFO - Chain [1] done processing
19:02:30 - cmdstanpy - INFO - Chain [1] start processing
19:02:30 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stat

  ✓  site_based/2_model_performance_dg_units_in_kwh_site_based.csv
  ✓  site_based/2_forecast_long_dg_units_in_kwh_site_based.csv
  ✓  site_based/2_forecast_best_model_dg_units_in_kwh_site_based.csv
  ✓  site_based/2_forecast_all_models_dg_units_in_kwh_site_based.csv
  ▸ target: total_units_kwh
    preprocessed rows : 3,575


19:02:38 - cmdstanpy - INFO - Chain [1] start processing
19:02:38 - cmdstanpy - INFO - Chain [1] done processing
19:02:38 - cmdstanpy - INFO - Chain [1] start processing
19:02:38 - cmdstanpy - INFO - Chain [1] done processing
19:02:39 - cmdstanpy - INFO - Chain [1] start processing
19:02:39 - cmdstanpy - INFO - Chain [1] done processing
19:02:39 - cmdstanpy - INFO - Chain [1] start processing
19:02:39 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
19:02:39 - cmdstanpy - INFO - Chain [1] start processing
19:0

## Power BI — How to Load the CSVs

1. **Get Data → Folder** — point to each subfolder (`state_based`, `city_based`, `site_based`) to load all CSVs at once, or import individually via **Text/CSV**.
2. Use `target` column as a slicer to switch between the three emission metrics.
3. Use `indicator_name` as a slicer to filter by energy indicator.
4. Set the `date` column data type to **Date** in Power Query.
5. Key columns for visuals:
   - Dimensions: `state`, `city`, `site`, `indicator_name`, `target`, `model`, `series` / `series_type`, `best_model`
   - Measures: `value`, `lower_ci`, `upper_ci`, `MAE`, `RMSE`, `MAE_%`, `RMSE_%`, `is_best`
